# Companies House CRM Matching

End-to-end pipeline:

1. read `crm_companies.csv`
2. clean company names
3. search candidates on Companies House (`GET /search/companies`)
4. pick the most plausible match
5. enrich matched records (`GET /company/{companyNumber}`)
6. save `outputs/company_matches_enriched.csv`

Docs: https://developer-specs.company-information.service.gov.uk/companies-house-public-data-api/reference

In [1]:
from __future__ import annotations

import getpass
import os
import re
import time
import unicodedata
from difflib import SequenceMatcher
from pathlib import Path

import pandas as pd
import requests

# ---------------------------------------------------------------- config
LIVE_BASE = "https://api.company-information.service.gov.uk"
SANDBOX_BASE = "https://api-sandbox.company-information.service.gov.uk"

API_BASE = LIVE_BASE          
INPUT_PATH = Path("crm_companies.csv")
OUTPUT_PATH = Path("outputs/company_matches_enriched.csv")

API_KEY = os.environ.get("COMPANIES_HOUSE_API_KEY", "").strip()
if not API_KEY:
    API_KEY = getpass.getpass("Companies House REST API key: ").strip()

print("input exists:", INPUT_PATH.exists())
print("api key set:", bool(API_KEY), f"(length {len(API_KEY)})")
print("api base:", API_BASE)

input exists: True
api key set: True (length 36)
api base: https://api.company-information.service.gov.uk


In [ ]:
class CHAuthError(RuntimeError):
    """401/403 — the key itself is being rejected. Retrying is pointless."""

class CHRequestError(RuntimeError):
    """Non-auth request failure that persisted after retries."""

class RateLimiter:
    """Companies House allows 600 requests per 5-minute window.
    Pace to ~1.8 req/s to stay comfortably under it."""
    def __init__(self, min_interval: float = 0.55):
        self.min_interval = min_interval
        self._last = 0.0

    def wait(self):
        delta = time.monotonic() - self._last
        if delta < self.min_interval:
            time.sleep(self.min_interval - delta)
        self._last = time.monotonic()


SESSION = requests.Session()
SESSION.auth = (API_KEY, "")         
SESSION.headers.update({"User-Agent": "crm-company-matcher/2.0"})
LIMITER = RateLimiter()


def ch_get_json(path: str, params: dict | None = None, max_retries: int = 4) -> dict:
    """GET {API_BASE}{path} with retries for transient failures.

    Raises CHAuthError on 401/403 (fail fast — abort the whole run).
    Raises CHRequestError if still failing after retries.
    """
    url = f"{API_BASE}{path}"
    last_err = ""
    for attempt in range(1, max_retries + 1):
        LIMITER.wait()
        try:
            resp = SESSION.get(url, params=params, timeout=30)
        except requests.RequestException as exc:
            last_err = f"network error: {exc}"
        else:
            if resp.status_code == 200:
                return resp.json()
            if resp.status_code in (401, 403):
                raise CHAuthError(
                    f"HTTP {resp.status_code} from {url} — API key rejected. "
                    f"Body: {resp.text[:200]}"
                )
            if resp.status_code == 404:
                return {}          
            if resp.status_code == 429:
                retry_after = int(resp.headers.get("Retry-After", "5") or 5)
                print(f"  rate limited (429); sleeping {retry_after}s")
                time.sleep(retry_after)
                continue
            last_err = f"HTTP {resp.status_code}: {resp.text[:200]}"
            if resp.status_code < 500:
                break             
        time.sleep(min(2 ** attempt, 10))
    raise CHRequestError(f"GET {url} failed after {max_retries} attempts — {last_err}")


def search_companies(query: str, items_per_page: int = 10) -> list[dict]:
    data = ch_get_json("/search/companies", {"q": query, "items_per_page": items_per_page})
    return data.get("items", [])


def get_company_profile(company_number: str) -> dict:
    return ch_get_json(f"/company/{company_number}")

In [4]:
def strip_accents(text: str) -> str:
    normalized = unicodedata.normalize("NFKD", text)
    return "".join(ch for ch in normalized if not unicodedata.combining(ch))


def normalize_text(text: str) -> str:
    text = strip_accents(text or "")
    text = text.lower().strip()
    text = text.replace("&", " and ")
    text = re.sub(r"www\.", "", text)
    text = re.sub(r"[^a-z0-9]+", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def clean_company_name(name: str) -> str:
    if not name:
        return ""
    text = strip_accents(name)
    text = text.replace("&", " and ")
    text = re.sub(r"^\s*www\.", "", text, flags=re.I)
    text = re.sub(r"\([^)]*\)", " ", text)
    text = re.sub(r"[^A-Za-z0-9]+", " ", text)
    bad = {"ltd", "limited", "plc", "inc", "llp", "lp", "co", "company", "the", "uk", "ukltd"}
    tokens_ = [tok for tok in text.split() if tok.lower() not in bad]
    return " ".join(tokens_).strip()


def build_query_variants(name: str) -> list[str]:
    variants: list[str] = []
    raw = (name or "").strip()
    if not raw:
        return variants
    variants.append(raw)
    cleaned = clean_company_name(raw)
    if cleaned and cleaned not in variants:
        variants.append(cleaned)
    for match in re.findall(r"\(([^)]+)\)", raw):
        match = match.strip()
        if match and len(match) <= 12 and match not in variants:
            variants.append(match)
    if "(" in raw:
        before = clean_company_name(raw.split("(", 1)[0].strip())
        if before and before not in variants:
            variants.append(before)
    if raw.lower().startswith("www."):
        host = raw[4:].strip()
        if host and host not in variants:
            variants.append(host)
    return list(dict.fromkeys(variants))


def tokens(text: str) -> list[str]:
    return [t for t in normalize_text(text).split() if t]


def acronym(text: str) -> str:
    stop = {"ltd", "limited", "plc", "inc", "llp", "lp", "co", "company", "the", "uk", "ukltd"}
    return "".join(t[0] for t in tokens(text) if t not in stop)


def score_pair(source_name: str, candidate_name: str) -> tuple[float, str]:
    src = normalize_text(source_name)
    cand = normalize_text(candidate_name)
    if not src or not cand:
        return 0.0, "empty"
    if src == cand:
        return 1.0, "exact match"

    src_tokens = tokens(source_name)
    cand_tokens = tokens(candidate_name)
    overlap = len(set(src_tokens) & set(cand_tokens)) if src_tokens and cand_tokens else 0
    token_score = (2 * overlap) / (len(set(src_tokens)) + len(set(cand_tokens))) if src_tokens and cand_tokens else 0.0
    seq_score = SequenceMatcher(None, src, cand).ratio()

    src_acr = acronym(source_name)
    cand_acr = acronym(candidate_name)
    acr_score = 1.0 if src_acr and cand_acr and src_acr == cand_acr else 0.0
    if not acr_score and src_acr and src_acr in cand.replace(" ", ""):
        acr_score = 0.8
    elif not acr_score and cand_acr and cand_acr in src.replace(" ", ""):
        acr_score = 0.8

    bonus = 0.12 if src in cand or cand in src else 0.0
    score = min((0.55 * token_score) + (0.35 * seq_score) + (0.10 * acr_score) + bonus, 1.0)
    reason = f"token={token_score:.2f}, sequence={seq_score:.2f}"
    if acr_score:
        reason += f", acronym={acr_score:.2f}"
    if bonus:
        reason += ", substring bonus"
    return score, reason


def flatten_address(address: dict) -> str:
    if not address:
        return ""
    parts = [
        address.get("address_line_1"),
        address.get("address_line_2"),
        address.get("locality"),
        address.get("region"),
        address.get("postal_code"),
        address.get("country"),
    ]
    return ", ".join(part for part in parts if part)

In [ ]:
LEGAL_TOKENS = {"ltd", "limited", "plc", "llp", "lp", "inc", "co", "company",
                "holdings", "group", "ag", "ab", "bv", "sa", "gmbh", "nv", "oy",
                "as", "uk", "the"}

def core_tokens(text: str) -> list[str]:
    """Tokens with legal suffixes removed — applied to BOTH names before scoring."""
    return [w for w in normalize_text(text).split() if w not in LEGAL_TOKENS]

def core_string(text: str) -> str:
    return " ".join(core_tokens(text))

def score_pair(source_name: str, candidate_name: str) -> tuple[float, str]: 
    src = core_string(source_name)
    cand = core_string(candidate_name)
    if not src or not cand:
        return 0.0, "empty"
    if src == cand:
        return 1.0, "exact core match (legal suffix ignored)"

    st, ct = src.split(), cand.split()
    overlap = len(set(st) & set(ct))
    token_score = (2 * overlap) / (len(set(st)) + len(set(ct))) if st and ct else 0.0
    seq_score = SequenceMatcher(None, src, cand).ratio()

    exact_word = 1.0 if (len(st) <= 2 and all(w in ct for w in st)) else 0.0

    src_acr, cand_acr = acronym(source_name), acronym(candidate_name)
    acr_score = 1.0 if src_acr and cand_acr and src_acr == cand_acr else 0.0

    bonus = 0.15 if (src in cand or cand in src) else 0.0
    score = min(0.45 * token_score + 0.30 * seq_score + 0.25 * exact_word
                + 0.05 * acr_score + bonus, 1.0)

    reason = f"token={token_score:.2f}, seq={seq_score:.2f}"
    if exact_word: reason += ", exact-word"
    if acr_score:  reason += ", acronym"
    if bonus:      reason += ", substring"
    return score, reason

for q, c in [("Deliveroo", "DELIVEROO LIMITED"), ("Ohme", "OHME LTD"),
             ("Zapmap", "ZAPMAP LIMITED"), ("Oxa", "OXA AUTONOMY LTD")]:
    print(f"{q:<12} vs {c:<22} -> {score_pair(q, c)[0]:.2f}")

Deliveroo    vs DELIVEROO LIMITED      -> 1.00
Ohme         vs OHME LTD               -> 1.00
Zapmap       vs ZAPMAP LIMITED         -> 1.00
Oxa          vs OXA AUTONOMY LTD       -> 0.82


In [6]:
KNOWN_ALIASES = {
    "wise (formerly transferwise)": ["Wise", "TransferWise"],
    "oxa": ["Oxa", "Oxbotica"],
    "the hut group (thg)": ["The Hut Group", "THG"],
}

KNOWN_FOREIGN = {
    "dott b.v.": "Netherlands",
    "voi technology ab": "Sweden",
    "northvolt ab": "Sweden",
    "flixbus": "Germany",
}

def extra_variants(raw_name: str) -> list[str]:
    return KNOWN_ALIASES.get(str(raw_name).strip().lower(), [])

def foreign_note(raw_name: str) -> str:
    country = KNOWN_FOREIGN.get(str(raw_name).strip().lower())
    return f"Known {country} entity; UK register likely has no primary record" if country else ""

_base_build_query_variants = build_query_variants
def build_query_variants(name: str) -> list[str]:         
    variants = _base_build_query_variants(name)
    for alias in extra_variants(name):
        if alias not in variants:
            variants.append(alias)
    return variants

print("Alias/foreign overrides loaded.")
print("  Ohme ->", build_query_variants("Ohme"))
print("  Wise ->", build_query_variants("Wise (formerly TransferWise)"))
print("  Dott ->", build_query_variants("Dott B.V."), "| note:", foreign_note("Dott B.V."))

Alias/foreign overrides loaded.
  Ohme -> ['Ohme']
  Wise -> ['Wise (formerly TransferWise)', 'Wise', 'TransferWise']
  Dott -> ['Dott B.V.', 'Dott B V'] | note: Known Netherlands entity; UK register likely has no primary record


In [ ]:
def pick_match(input_name: str, hq_country: str) -> tuple[dict | None, list[dict], str, str]:
    """Returns (chosen, scored_candidates, search_term_used, api_error).

    CHAuthError propagates (aborts the run). Transient failures that survive
    retries are reported in api_error instead of being hidden as "no results".
    """
    variants = build_query_variants(input_name)
    all_candidates: dict[str, dict] = {}
    search_term_used = variants[0] if variants else ""
    api_errors: list[str] = []

    for variant in variants:
        try:
            results = search_companies(variant)
        except CHRequestError as exc:
            api_errors.append(f"'{variant}': {exc}")
            continue
        if results:
            search_term_used = variant
        for item in results:
            num = item.get("company_number")
            if num and num not in all_candidates:
                all_candidates[num] = item

    scored = []
    for item in all_candidates.values():
        best_score = 0.0
        best_reason = ""
        for variant in variants:
            score, reason = score_pair(variant, item.get("title", ""))
            if score > best_score:
                best_score = score
                best_reason = f"query='{variant}' -> {reason}"
        if (hq_country or "").strip().lower() != "united kingdom":
            best_score = max(0.0, best_score - 0.05)
            best_reason += "; non-UK HQ penalty"
        scored.append({
            "company_name": item.get("title", ""),
            "company_number": item.get("company_number", ""),
            "company_status": item.get("company_status", ""),
            "company_type": item.get("company_type", ""),
            "date_of_creation": item.get("date_of_creation", ""),
            "address_snippet": item.get("address_snippet", ""),
            "score": best_score,
            "reason": best_reason,
        })

    scored.sort(key=lambda x: x["score"], reverse=True)
    api_error = " | ".join(api_errors)
    if not scored:
        return None, [], search_term_used, api_error

    top = scored[0]
    second = scored[1] if len(scored) > 1 else None
    second_score = second["score"] if second else 0.0

    def _same_entity(a, b):
        if not a or not b:
            return False
        return score_pair(a["company_name"], b["company_name"])[0] >= 0.90

    margin = top["score"] - second_score
    if top["score"] >= 0.82 and margin >= 0.05:
        chosen = top                                 
    elif top["score"] >= 0.93 and _same_entity(top, second):
        chosen = top                                  
        top["reason"] += "; top-2 are same entity, picked highest"
    else:
        chosen = None                                
    return chosen, scored, search_term_used, api_error

In [8]:
df_in = pd.read_csv(INPUT_PATH)
df_in.info()
df_in.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 58 entries, 0 to 57
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   crm_id        58 non-null     object
 1   company_name  56 non-null     object
 2   sector        58 non-null     object
 3   hq_country    58 non-null     object
dtypes: object(4)
memory usage: 1.9+ KB


,crm_id,company_name,sector,hq_country
0,CRM-0001,Ohme,Transport & Mobility,United Kingdom
1,CRM-0002,Alexandr Dennis,Transport & Mobility,United Kingdom
2,CRM-0003,Dott B.V.,Transport & Mobility,Netherlands
3,CRM-0004,Octopus Energy Ltd,Energy,United Kingdom
4,CRM-0005,Innocent Drinks,Food & Beverage,United Kingdom


In [ ]:
import re

LEGAL_SUFFIXES = r"\b(ltd|limited|plc|llp|lp|inc|co|company|holdings|group|ag|a\.?b\.?|b\.?v\.?|s\.?a\.?|gmbh|nv|oy|as)\b"
FOREIGN_SUFFIX = r"\b(a\.?b\.?|b\.?v\.?|gmbh|s\.?a\.?|nv|oy|as|ag)\b"  # non-UK 

def profile_name(name: str) -> list[str]:
    tags = []
    raw = str(name or "").strip()
    if not raw or raw.lower() in {"nan", "n/a", ""}:
        return ["blank"]
    low = raw.lower()
    if raw.lower().startswith("www.") or re.search(r"\.[a-z]{2,}$", low):
        tags.append("url_as_name")
    if "(" in raw and ")" in raw:
        tags.append("parenthetical")
    if "formerly" in low or "aka" in low or "now " in low:
        tags.append("renamed")
    if re.search(FOREIGN_SUFFIX, low):
        tags.append("foreign_legal_form")
    if re.search(LEGAL_SUFFIXES, low):
        tags.append("has_legal_suffix")
    if " - " in raw or re.search(r"\b(uk|cambridge|london|oxford)\b", low):
        tags.append("location_or_country_noise")
    generic = {"solutions", "logistics", "mobility", "transport", "energy",
               "systems", "global", "premier", "smart", "green", "city", "connect"}
    words = set(re.sub(r"[^a-z ]", " ", low).split())
    if len(words & generic) >= 1 and len(words) <= 3:
        tags.append("possibly_generic")
    return tags or ["clean"]

df_in["messiness_tags"] = df_in["company_name"].apply(profile_name)

from collections import Counter
tag_counts = Counter(t for tags in df_in["messiness_tags"] for t in tags)
print("Messiness patterns across", len(df_in), "rows:")
for tag, n in tag_counts.most_common():
    print(f"  {n:2d}  {tag}")

n_blank = df_in["company_name"].isna().sum()
print(f"\nBlank/null company names: {n_blank}")
df_in[["crm_id", "company_name", "messiness_tags"]].head(15)

Messiness patterns across 58 rows:
  30  has_legal_suffix
  18  clean
  11  possibly_generic
   3  foreign_legal_form
   2  parenthetical
   2  location_or_country_noise
   2  blank
   1  renamed
   1  url_as_name

Blank/null company names: 2


,crm_id,company_name,messiness_tags
0,CRM-0001,Ohme,[clean]
1,CRM-0002,Alexandr Dennis,[clean]
2,CRM-0003,Dott B.V.,"[foreign_legal_form, has_legal_suffix]"
3,CRM-0004,Octopus Energy Ltd,"[has_legal_suffix, possibly_generic]"
4,CRM-0005,Innocent Drinks,[clean]
5,CRM-0006,The Hut Group (THG),"[parenthetical, has_legal_suffix]"
6,CRM-0007,EO Charging,[clean]
7,CRM-0008,Octopus Energy,[possibly_generic]
8,CRM-0009,Carillion plc,[has_legal_suffix]
9,CRM-0010,Oxbotica,[clean]


In [12]:
output_rows = []
started = time.monotonic()

for i, row in df_in.iterrows():
    raw_name = str(row.get("company_name", "")).strip()
    hq_country = str(row.get("hq_country", "")).strip()

    if not raw_name or raw_name.lower() in {"n/a", "nan"}:
        continue

    cleaned_name = clean_company_name(raw_name)
    chosen, candidates, search_term_used, api_error = pick_match(raw_name, hq_country)

    matched_profile: dict = {}
    if not candidates:
        confidence = "api_error" if api_error else "no_match"
        score_str = "0.00"
        match_reason = api_error or "No search results returned"
    else:
        top = candidates[0]
        second = candidates[1] if len(candidates) > 1 else None
        if chosen is None:
            if top["score"] < 0.65:
                confidence = "no_match"
                match_reason = f"Best score too low ({top['reason']})"
            elif second and abs(top["score"] - second["score"]) < 0.05:
                confidence = "ambiguous"
                match_reason = "Top two candidates are too close"
            else:
                confidence = "no_match"
                match_reason = "No confident match"
            score_str = f"{top['score']:.2f}"
        else:
            if chosen["score"] >= 0.93:
                confidence = "high"
            elif chosen["score"] >= 0.85:
                confidence = "medium"
            else:
                confidence = "low"
            score_str = f"{chosen['score']:.2f}"
            match_reason = chosen["reason"]
            if second:
                match_reason += f"; runner-up={second['company_name']} ({second['score']:.2f})"
            try:
                matched_profile = get_company_profile(chosen["company_number"])
            except CHRequestError as exc:
                api_error = (api_error + " | " if api_error else "") + f"profile: {exc}"

    address = flatten_address(matched_profile.get("registered_office_address", {})) if matched_profile else ""
    if not address and chosen:
        address = chosen.get("address_snippet", "")

    notes = []
    if (row.get("hq_country") or "").strip().lower() != "united kingdom":
        notes.append(f"HQ country is {row.get('hq_country')}; UK register may have no exact match")
    if confidence in {"low", "no_match", "ambiguous", "api_error"}:
        notes.append("Review manually")
    if chosen is None and candidates:
        notes.append("No confident winner selected")

    output_rows.append({
        "crm_id": row.get("crm_id", ""),
        "input_company_name": raw_name,
        "cleaned_company_name": cleaned_name,
        "matched_company_name": matched_profile.get("company_name", chosen["company_name"] if chosen else ""),
        "companies_house_number": chosen["company_number"] if chosen else "",
        "company_status": matched_profile.get("company_status", chosen["company_status"] if chosen else ""),
        "company_type": matched_profile.get("type", chosen["company_type"] if chosen else ""),
        "date_of_creation": matched_profile.get("date_of_creation", chosen["date_of_creation"] if chosen else ""),
        "registered_address": address,
        "match_confidence": confidence,
        "match_score": score_str,
        "match_reason": match_reason,
        "api_error": api_error,
        "notes": " | ".join(notes),
        "search_term_used": search_term_used,
    })

    if i % 10 == 0:
        elapsed = time.monotonic() - started
        print(f"processed {i}/{len(df_in)}  ({elapsed:.0f}s elapsed)")

print("loop finished:", len(output_rows), "rows")

processed 0/58  (1s elapsed)
processed 10/58  (12s elapsed)
processed 20/58  (31s elapsed)
processed 30/58  (42s elapsed)
processed 40/58  (58s elapsed)
processed 50/58  (71s elapsed)
loop finished: 56 rows


In [13]:
df_out = pd.DataFrame(output_rows)
print(df_out["match_confidence"].value_counts(dropna=False))
df_out.head(10)

match_confidence
high         42
no_match     10
low           2
ambiguous     2
Name: count, dtype: int64


,crm_id,input_company_name,cleaned_company_name,matched_company_name,companies_house_number,company_status,company_type,date_of_creation,registered_address,match_confidence,match_score,match_reason,api_error,notes,search_term_used
0,CRM-0001,Ohme,Ohme,OHME OPERATIONS UK LIMITED,11933185,active,ltd,2019-04-08,"5th Floor, Wellington House, 125 - 130 Strand,...",low,0.83,"query='Ohme' -> token=0.67, seq=0.42, exact-wo...",,Review manually,Ohme
1,CRM-0002,Alexandr Dennis,Alexandr Dennis,,,,,,,no_match,0.57,Best score too low (query='Alexandr Dennis' ->...,,Review manually | No confident winner selected,Alexandr Dennis
2,CRM-0003,Dott B.V.,Dott B V,,,,,,,no_match,0.26,Best score too low (query='Dott B.V.' -> token...,,HQ country is Netherlands; UK register may hav...,Dott B V
3,CRM-0004,Octopus Energy Ltd,Octopus Energy,OCTOPUS ENERGY LIMITED,09263424,active,ltd,2014-10-14,"Uk House, 5th Floor, 164-182 Oxford Street, Lo...",high,1.00,query='Octopus Energy Ltd' -> exact core match...,,,Octopus Energy
4,CRM-0005,Innocent Drinks,Innocent Drinks,,,,,,,no_match,0.66,No confident match,,Review manually | No confident winner selected,Innocent Drinks
5,CRM-0006,The Hut Group (THG),Hut Group,,,,,,,ambiguous,1.00,Top two candidates are too close,,Review manually | No confident winner selected,The Hut Group
6,CRM-0007,EO Charging,EO Charging,,,,,,,ambiguous,0.95,Top two candidates are too close,,Review manually | No confident winner selected,EO Charging
7,CRM-0008,Octopus Energy,Octopus Energy,OCTOPUS ENERGY LIMITED,09263424,active,ltd,2014-10-14,"Uk House, 5th Floor, 164-182 Oxford Street, Lo...",high,1.00,query='Octopus Energy' -> exact core match (le...,,,Octopus Energy
8,CRM-0009,Carillion plc,Carillion,CARILLION PLC,03782379,liquidation,plc,1999-05-28,"Central Square 8th Floor, 29 Wellington Street...",high,1.00,query='Carillion plc' -> exact core match (leg...,,,Carillion
9,CRM-0010,Oxbotica,Oxbotica,,,,,,,no_match,0.78,No confident match,,Review manually | No confident winner selected,Oxbotica


In [ ]:
df_out.to_csv(OUTPUT_PATH, index=False)